In [1]:
import datasets, subprocess
from utils import GrazieProvider, ManimProvider, extract_json
from grazie.api.client.gateway import AuthType, GrazieApiGatewayClient, GrazieAgent
from grazie.api.client.endpoints import GrazieApiGatewayUrls

In [2]:
with open("./token.secret", 'r') as t: token = t.read()

client = GrazieApiGatewayClient(
    grazie_agent=GrazieAgent(name="grazie-api-gateway-client-readme", version="dev"),
    url=GrazieApiGatewayUrls.STAGING,
    grazie_jwt_token=token,
    auth_type=AuthType.USER,
)

In [3]:
provider = GrazieProvider(client, model="openai-gpt-4o")
ds = datasets.load_from_disk("subset")

In [5]:
term = ds[0]
metaphor = term["metaphor"]
term

{'value': 'Boolean',
 'definition': 'A data type that has one of two possible values (usually denoted true and false) intended to represent the two truth values of logic and Boolean algebra.',
 'metaphor': 'Imagine a light switch in your room. This switch can only be in one of two positions: ON or OFF.\n\n- When the switch is ON, the light is shining (true).\n- When the switch is OFF, the light is not shining (false).\n\nIn programming, a Boolean is like this light switch. It can only be in one of two states: true (ON) or false (OFF). Just as the light switch controls whether the light is on or off, a Boolean controls whether a condition is true or false.'}

In [6]:
classes = provider.get_classes(term, metaphor)

In [8]:
classes_dict = extract_json(classes)

In [9]:
# Static analysis with pylint for each element

for i in range(len(classes_dict['elements'])):
    open("/tmp/dummy.py", "w").write(
        f"{classes_dict['elements'][i]['code']}"
    )
    command = ["pylint", "-E", "/tmp/dummy.py"]
    process = subprocess.run(command, capture_output=True, text=True)
    static_errors = process.stdout
    print(f"{i}: {static_errors}")

0: 
1: 
2: ************* Module dummy
/tmp/dummy.py:8:28: E0602: Undefined variable 'LightSwitch' (undefined-variable)
/tmp/dummy.py:11:21: E0602: Undefined variable 'Light' (undefined-variable)



In [10]:
desc = provider.get_description(term, metaphor, str(classes_dict))
# desc = ""
manim_code = provider.get_manim(term, metaphor, str(classes_dict), desc)

In [14]:
manim_provider = ManimProvider(provider, term, 
                               executable="/home/ynoviello/anaconda3/envs/jetbrains/bin", 
                               working_dir="./manim_stuff")

In [15]:
manim_provider.write_python(manim_code)
error = manim_provider.execute_manim_script()
print(error)

100


In [51]:
# import json
# with open("/home/ynoviello/PycharmProjects/AI_Metaphors/manim_stuff/scripts/best-scripts/replace.json", 'w') as j:
#     json.dump(classes_dict, j)

In [11]:
error = manim_provider.fix_code(error)
print(error)